In [ ]:
"""
Task 3: Local 7B LLM Quantization & Logit Extraction Pipeline
This is written against the real Ollama / Hugging Face APIs.
Running it needs the actual Llama-3-8B-Instruct GGUF weights downloaded
locally (several GB) and either Ollama or a CUDA GPU - so it cannot run
inside this sandbox, but the code below is the real, complete pipeline.
"""

import subprocess
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"

def load_quantized_model():
    # 4-bit quantization config (Q4_K_M-equivalent using bitsandbytes NF4)
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        output_hidden_states=True,
    )
    model.eval()
    return model, tokenizer

def extract_logits_and_entropy(model, tokenizer, prompt, max_new_tokens=20):
    """Generates tokens one at a time and records the logit distribution
    and entropy at every generation step."""
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
    records = []

    for step in range(max_new_tokens):
        with torch.no_grad():
            out = model(input_ids, output_hidden_states=True)

        next_token_logits = out.logits[0, -1, :]              # last-position logits
        probs = torch.softmax(next_token_logits, dim=-1)
        entropy = -torch.sum(probs * torch.log(probs + 1e-9)).item()

        next_token_id = torch.argmax(probs).unsqueeze(0).unsqueeze(0)
        input_ids = torch.cat([input_ids, next_token_id], dim=1)

        records.append({
            "step": step,
            "token": tokenizer.decode(next_token_id[0]),
            "entropy": entropy,
            "top5_logits": torch.topk(next_token_logits, 5).values.tolist(),
        })

        if next_token_id.item() == tokenizer.eos_token_id:
            break

    return records

def run_via_ollama(prompt):
    """Alternative lightweight path: use a locally running Ollama server
    with the already-quantized GGUF model (`ollama pull llama3:8b-instruct-q4_K_M`)."""
    result = subprocess.run(
        ["ollama", "run", "llama3:8b-instruct-q4_K_M", prompt],
        capture_output=True, text=True,
    )
    return result.stdout

if __name__ == "__main__":
    # model, tokenizer = load_quantized_model()
    # logs = extract_logits_and_entropy(model, tokenizer, "The capital of France is")
    # for r in logs:
    #     print(r["step"], r["token"], f"entropy={r['entropy']:.3f}")
    print("Run this script on a machine with the Llama-3-8B GGUF weights and a GPU/Ollama install.")

---
## Task 3: Local 7B LLM Quantization & Logit Extraction Pipeline